## Without web search

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI

load_dotenv()

True

In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model=AzureChatOpenAI(azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"])
)

In [3]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [4]:
print(response['messages'][-1].content)

My training knowledge goes up to **June 2024**. I may not know about developments after that unless you provide the information or I use tools to check current sources.


## Add web search tool

For deployments that reject function tools with reasoning on Chat Completions, set `reasoning_effort="none"` explicitly. Omitting the parameter or using Python `None` leaves the model's default reasoning behavior in place.

In [5]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://localwiki.org/sf/Mayor',
   'title': 'Mayor - San Francisco - LocalWiki',
   'score': 0.94263905,
   'content': 'Image 1\n\nlog in or create account\n\n   Explore\n   About\n   Blog\n   Donate\n   API\n   Help\n\n# San Francisco\n\n[](\n   Explore\n   Map\n   Activity\n\nAdd a new page\n\n   Edit\n   Info\n\n# Mayor\n\nThe current Mayor of the City is Ed Lee. He was elected using a modified Instant Runoff Voting system.\n\nMayor Ed Lee was chosen by Gavin Newsom to complete his term. Lee then won to become 43rd mayor of San Francisco. He was elected using a modified Instant Runoff Voting system. [...] Famous prior mayors include Willie Brown, who served for many years as Speaker of the California Assembly prior to being mayor, and Dianne Feinstein, current U.S. Senator.\n\nWikipedia has a list of all the mayors if someone wants to get it.\n

In [6]:
agent = create_agent(
    model=AzureChatOpenAI(
        azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
        reasoning_effort="none",
    ),
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

BadRequestError: Error code: 400 - {'error': {'message': "Function tools with reasoning_effort are not supported for this model in /v1/chat/completions. To use function tools, use /v1/responses or set reasoning_effort to 'none'.", 'type': 'invalid_request_error', 'param': 'reasoning_effort', 'code': None}}

In [7]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='How up to date is your training knowledge?', additional_kwargs={}, response_metadata={}, id='907d0cd2-55b0-4adb-a544-718227a51359'),
 AIMessage(content='My training knowledge goes up to **June 2024**. I may not know about developments after that unless you provide the information or I use tools to check current sources.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 15, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 17, 'engine_ttft_ms': 307, 'engine_ttlt_ms': 989, 'pre_inference_ms': 83, 'service_tbt_ms': 18, 'service_ttft_ms': 557, 'service_ttlt_ms': 1232, 'user_visible_ttft_ms': 474}}, 'model_provider': 'openai', 'model_name': 'gpt-6.1-sol-2026-09-29', '

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r